## Immutable ConfigMaps & Secrets

By default the kubelet **watches** every ConfigMap a pod mounts so it can notice changes. On a large cluster — thousands of ConfigMaps × pods × nodes — that's a lot of watches.

Setting `immutable: true` tells the kubelet **"this never changes — don't watch it."** Once you flip the bit, the object can only be **deleted and recreated**, not edited.

```yaml
apiVersion: v1
kind: ConfigMap
metadata: { name: app-config-v3 }   # version in the name — you can't edit it
data: { LOG_LEVEL: warn }
immutable: true
```

Two payoffs: **performance** (no watch) and **safety** (a config that can't be mutated out from under a running release). The same field exists on Secrets.

The typical pattern is **release-versioned config**: ship `app-config-v3` alongside Deployment `web-v3`, then delete the pair when you tear that version down — the config is born, deployed, and dies with one release, never patched. Tools like **Kustomize and Helm** append a hash of the data to the name automatically, so every change effectively mints a new immutable object (and, conveniently, forces a rollout because the Deployment now references a new name).

The trade-off: you lose live-update — but for release-scoped config that's exactly what you want. On our map this is a property of the **ConfigMap** and **Secret** chips in the Config box: the same objects, pinned so nothing edits them mid-flight.